# Spectral Synthesis — documentation examples

Companion notebook to the **Spectral Synthesis** documentation page
(`docs/source/spectral_synthesis.rst`). One section per code snippet.

Requires only the core `stellar-spice` install.

In [1]:
%matplotlib inline
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")

import warnings
from pathlib import Path
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

DOCS_IMG = Path("..") / ".." / "docs" / "img"

def save_doc_fig(name, make_fig):
    """Save make_fig() as <name>.png (light) and <name>_dark.png in docs/img."""
    for style, suffix in [("default", ""), ("dark_background", "_dark")]:
        with plt.style.context(style):
            fig = make_fig()
            fig.savefig(DOCS_IMG / f"{name}{suffix}.png", dpi=120, bbox_inches="tight",
                        facecolor=fig.get_facecolor())
            plt.close(fig)
    print(f"saved {name}.png / {name}_dark.png")

## Basic usage

In [2]:
from spice.models import IcosphereModel
from spice.spectrum import simulate_observed_flux, Blackbody

bb = Blackbody()
star = IcosphereModel.construct(1000, 1., 1., bb.solar_parameters, bb.parameter_names)

wavelengths = np.linspace(4000., 7000., 2000)   # Angstroms
flux = simulate_observed_flux(bb.intensity, star, np.log10(wavelengths))

print("shape:", flux.shape, "| peak flux:", float(flux[:, 0].max()), "erg/s/cm^2/A at 10 pc")

[spice] IcosphereModel constructed in 0.8 s


shape: (2000, 2) | peak flux: 4.211370723932098e-11 erg/s/cm^2/A at 10 pc


## Parameters that matter: `ld_law` / `ld_coeffs`

`Blackbody().intensity` does not accept the `ld_law`/`ld_coeffs` keywords, so
passing them to `simulate_observed_flux` raises `TypeError`; give the law to
the `Blackbody` constructor instead.

In [3]:
try:
    simulate_observed_flux(bb.intensity, star, np.log10(wavelengths),
                           ld_law="linear", ld_coeffs=[0.5])
except TypeError as err:
    print("TypeError:", err)

bb_ld = Blackbody(ld_law="linear", ld_coeffs=[0.5])
flux_ld = simulate_observed_flux(bb_ld.intensity, star, np.log10(wavelengths))
print(f"limb-darkened / un-darkened flux at 5500 A: "
      f"{float(np.interp(5500., wavelengths, flux_ld[:, 0] / flux[:, 0])):.3f}")

TypeError: Blackbody.intensity() got an unexpected keyword argument 'ld_law'


limb-darkened / un-darkened flux at 5500 A: 0.833


## Doppler shifts: rotational broadening

A blackbody has no lines, so the page uses `GaussianLineEmulator` (blackbody
continuum + Gaussian absorption lines, core install only).

Figure: `rotation_broadening.png` / `rotation_broadening_dark.png`

In [4]:
from spice.models.mesh_transform import add_rotation, evaluate_rotation
from spice.spectrum import GaussianLineEmulator

emu = GaussianLineEmulator(line_centers=[5500.], line_widths=[0.1], line_depths=[0.6])
line_star = IcosphereModel.construct(1000, 1., 1., emu.to_parameters(), emu.parameter_names)
line_wavelengths = np.linspace(5497., 5503., 1000)   # Angstroms

flux_static = simulate_observed_flux(emu.intensity, line_star, np.log10(line_wavelengths))
fast = evaluate_rotation(add_rotation(line_star, rotation_velocity=50.), 0.)
flux_rot = simulate_observed_flux(emu.intensity, fast, np.log10(line_wavelengths))
normalized_rot = flux_rot[:, 0] / flux_rot[:, 1]   # lines / continuum

normalized_static = flux_static[:, 0] / flux_static[:, 1]
ew = lambda norm: float(np.trapezoid(1. - np.asarray(norm), line_wavelengths))
print(f"equivalent width: static {ew(normalized_static):.4f} A, 50 km/s {ew(normalized_rot):.4f} A")
print(f"v sin i / c x 5500 A = {50. / 299792.458 * 5500.:.3f} A")

def rotation_fig():
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(line_wavelengths, normalized_static, label="No rotation")
    ax.plot(line_wavelengths, normalized_rot, label="50 km/s")
    for edge in (-1, 1):
        ax.axvline(5500. * (1 + edge * 50. / 299792.458), color="0.5", lw=0.8, ls="--")
    ax.set_xlabel(r"Wavelength [$\AA$]")
    ax.set_ylabel("Normalized flux")
    ax.legend()
    return fig

save_doc_fig("rotation_broadening", rotation_fig)

PHOEBE: passband "Bolometric:900-40000" has a newer version available.  Run phoebe.list_passband_online_history("Bolometric:900-40000") to get a list of available changes and phoebe.update_passband("Bolometric:900-40000") or phoebe.update_all_passbands() to update.
[spice] IcosphereModel constructed in 0.0 s


equivalent width: static 0.1504 A, 50 km/s 0.1504 A
v sin i / c x 5500 A = 0.917 A
saved rotation_broadening.png / rotation_broadening_dark.png


## Distance scaling

Observed flux scales as $1/d^2$.

In [5]:
flux_10pc = simulate_observed_flux(bb.intensity, star, np.log10(wavelengths), distance=10.)
flux_100pc = simulate_observed_flux(bb.intensity, star, np.log10(wavelengths), distance=100.)

ratio = float(jnp.median(flux_10pc[:, 0] / flux_100pc[:, 0]))
print(f"flux(10 pc) / flux(100 pc) = {ratio:.1f} (expected 100)")

flux(10 pc) / flux(100 pc) = 100.0 (expected 100)


## Luminosities

`Blackbody.flux` returns the `mu=1` intensity, not a surface flux; for the
un-darkened blackbody the emergent surface flux is $\pi$ times it.

In [6]:
from spice.spectrum import (simulate_monochromatic_luminosity,
                            luminosity, absolute_bol_luminosity)

def surface_flux(log_wavelengths, parameters):
    return np.pi * bb.intensity(log_wavelengths, 1., parameters)

lum_wavelengths = np.geomspace(100., 1.e6, 4096)  # Angstroms
mono_lum = simulate_monochromatic_luminosity(
    surface_flux, star, np.log10(lum_wavelengths))  # erg/s/Angstrom
L = luminosity(surface_flux, star, lum_wavelengths)  # erg/s
M_bol = absolute_bol_luminosity(L)

print("mono_lum shape:", mono_lum.shape)
print(f"L = {float(L):.4e} erg/s ({float(L) / 3.828e33:.4f} L_sun), M_bol = {float(M_bol):.3f}")

mono_lum shape: (4096, 2)
L = 3.8451e+33 erg/s (1.0045 L_sun), M_bol = 4.735


## Instrumental resolution

The rotating line star from the Doppler section, convolved to a resolving
power that varies from 20000 to 30000 across the grid.

In [7]:
from spice.spectrum.utils import apply_variable_resolution

log_wavelengths = jnp.linspace(jnp.log10(5460.), jnp.log10(5540.), 2001)
flux_fast = simulate_observed_flux(emu.intensity, fast, log_wavelengths)
resolving_power = jnp.linspace(20000., 30000., len(log_wavelengths))
broadened = apply_variable_resolution(
    log_wavelengths, flux_fast[:, 0], resolving_power, segment_size=512)

print("broadened shape:", broadened.shape)
print(f"line-core depth: intrinsic {1 - float(jnp.min(flux_fast[:, 0] / flux_fast[:, 1])):.4f}, "
      f"broadened {1 - float(jnp.min(broadened / flux_fast[:, 1])):.4f}")

broadened shape: (2001,)
line-core depth: intrinsic 0.1043, broadened 0.1036


## Line-profile variations of non-radial pulsations

Residuals of a rotationally broadened line through one pulsation period: prograde bumps cross the profile blue → red, retrograde red → blue. The figure is written to `docs/img/` in light and dark variants.

In [8]:
from spice.models import IcosphereModel
from spice.models.mesh_transform import add_rotation, add_pulsation, evaluate_pulsations
from spice.spectrum.spectrum import simulate_observed_flux

emu = GaussianLineEmulator(line_centers=[5500.], line_widths=[0.05], line_depths=[0.6])
star = IcosphereModel.construct(8000, 2.0, 1.5, emu.to_parameters(), emu.parameter_names)
star = add_rotation(star, 60.)               # equator-on by default

period = 0.2                                 # days
wavelengths = jnp.linspace(5498.6, 5501.4, 600)
velocity = (np.asarray(wavelengths) / 5500. - 1.) * 299792.458
mode = jnp.array([[[0.01, 0.]], [[0.01, 0.]], [[0., 0.]]])   # radial + horizontal
phases = np.linspace(0., 1., 40, endpoint=False)

def residual_spectra(travelling):
    model = add_pulsation(star, 4, 4, period, mode, travelling=travelling)
    profiles = []
    for phase in phases:
        flux = simulate_observed_flux(emu.intensity,
                                      evaluate_pulsations(model, phase * period),
                                      jnp.log10(wavelengths))
        profiles.append(np.asarray(flux[:, 0] / flux[:, 1]))
    profiles = np.array(profiles)
    return profiles - profiles.mean(axis=0)

lpv = {sense: residual_spectra(sense) for sense in ["prograde", "retrograde"]}

[spice] IcosphereModel constructed in 0.6 s


[spice] Pulsations evaluated in 2.1 s


[spice] Pulsations evaluated in 0.2 s


[spice] Pulsations evaluated in 0.1 s


[spice] Pulsations evaluated in 0.1 s


[spice] Pulsations evaluated in 0.0 s


... (71 outputs left out)

[spice] Pulsations evaluated in 0.2 s


[spice] Pulsations evaluated in 0.4 s


[spice] Pulsations evaluated in 0.9 s


[spice] Pulsations evaluated in 0.3 s


In [9]:
def lpv_fig():
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
    vmax = max(np.abs(r).max() for r in lpv.values())
    for ax, (sense, residuals) in zip(axes, lpv.items()):
        im = ax.imshow(residuals, aspect="auto", origin="lower", cmap="RdBu_r",
                       vmin=-vmax, vmax=vmax,
                       extent=[velocity[0], velocity[-1], phases[0], phases[-1]])
        for v_edge in (-60, 60):
            ax.axvline(v_edge, color="0.5", lw=0.8, ls="--")
        ax.set_title(f"{sense.capitalize()} l = m = 4")
        ax.set_xlabel("Velocity [km/s]")
    axes[0].set_ylabel("Pulsation phase")
    fig.colorbar(im, ax=axes, label="Profile residual")
    return fig

save_doc_fig("travelling_mode_lpv", lpv_fig)

saved travelling_mode_lpv.png / travelling_mode_lpv_dark.png
